# Voice → Clean Text

A low-latency "clean up my speech" pipeline, close to # how Wispr Flow works:

1. **Whisper** (fine-tuned) — turns disfluent speech directly into clean text
2. **LLM** (fine-tuned) — cleans up text further and adapts tone (email / Slack / etc.)
3. **Custom CUDA kernel** — a hand-written fused op, benchmarked for training speed
4. **Merge + push to Hugging Face Hub** — so the Gradio app can load the trained models directly from the Hub

## 1. Install packages

In [ ]:
!pip install -q -U transformers>=4.44 accelerate peft bitsandbytes evaluate
!pip install -q -U librosa soundfile jiwer gTTS edge-tts
!pip install -q -U datasets huggingface_hub
!pip install -q -U torchao>=0.16.0

## 2. Check GPUs and Storage

In [ ]:
import torch
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

print("GPUs visible:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"cuda:{i} -> {torch.cuda.get_device_name(i)}")

In [ ]:
!nvidia-smi

In [ ]:
!df -h /kaggle/working 2>/dev/null || df -h .

## 3. Config

- `LLM_BASE` at 7B — best quality, still fast enough for a few hundred tokens of output once merged to fp16 (see Section 6).
- Switch `LLM_BASE` to `"Qwen/Qwen2.5-3B-Instruct"` — noticeably faster time-to-first-token and total generation time, small quality trade-off.

In [ ]:
# Whisper: distil-whisper is purpose-built for low-latency ASR
WHISPER_BASE = "distil-whisper/distil-small.en"

# LLM: 7B for quality (swap to "Qwen/Qwen2.5-3B-Instruct" for lower latency)
LLM_BASE = "Qwen/Qwen2.5-7B-Instruct"

# Where trained adapters get saved
WHISPER_ADAPTER_DIR = "whisper-lora-out"
LLM_ADAPTER_DIR = "llm-qlora-out"

# Where merged, serving-ready models get saved
WHISPER_MERGED_DIR = "whisper-merged"
LLM_MERGED_DIR = "llm-merged"

# Data settings
MAX_EXAMPLES = 1500
VAL_FRACTION = 0.05
CONCURRENCY = 16

# Training settings
WHISPER_EPOCHS = 50
WHISPER_BATCH_SIZE = 16
LLM_EPOCHS = 50
LLM_BATCH_SIZE = 4

## 4. Data preparation

Public disfluency / grammar-correction **text** dataset for two different purposes:

- **LLM cleanup training**: the `(noisy, clean)` text pairs, used
  directly — this is where the actual cleanup transformation should live.
- **Whisper training**: TTS-synthesized audio of the **noisy** sentence,
  paired with a transcript label that is *also* the noisy sentence (not
  the clean one). Whisper's only job is to transcribe **what was said,
  accurately**

In [ ]:
# (noisy_text, clean_text) pairs

pairs = []

# Load the Kaggle speech-cleanup-dataset
import kagglehub
from kagglehub import KaggleDatasetAdapter

DATASET_HANDLE = "bariankitvinod/speech-cleanup-dataset"
FILE_PATH = "speech_cleanup.parquet"

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    DATASET_HANDLE,
    FILE_PATH,
)

print("Loaded dataset:", DATASET_HANDLE)
print("Columns:", df.columns.tolist())
print("Rows:", len(df))

# Automatically identify noisy/clean columns
NOISY_COL = "noisy_text"
CLEAN_COL = "clean_text"

if NOISY_COL not in df.columns or CLEAN_COL not in df.columns:
    raise ValueError(
        f"Expected columns '{NOISY_COL}' and '{CLEAN_COL}', "
        f"but found: {df.columns.tolist()}"
    )

# Build (noisy_text, clean_text) pairs
for _, row in df.iterrows():
    noisy = row[NOISY_COL]
    clean = row[CLEAN_COL]

    # Some datasets may store multiple reference corrections as a list
    if isinstance(clean, list):
        clean = clean[0] if clean else None

    if noisy is None or clean is None:
        continue

    noisy = str(noisy).strip()
    clean = str(clean).strip()

    if not noisy or not clean:
        continue

    if noisy == clean:
        continue  # no cleanup signal, skip

    word_count = len(noisy.split())

    if word_count < 3 or word_count > 40:
        continue  # keep clips short so TTS + training stays fast

    pairs.append((noisy, clean))

print("Total combined pairs:", len(pairs))

In [ ]:
import random

random.seed(42)
random.shuffle(pairs)
pairs = pairs[:MAX_EXAMPLES]

num_val = max(1, int(len(pairs) * VAL_FRACTION))
val_pairs = pairs[:num_val]
train_pairs = pairs[num_val:]

print("Train pairs:", len(train_pairs))
print("Val pairs:", len(val_pairs))

In [ ]:
import json
import os

os.makedirs("data", exist_ok=True)

def write_jsonl(path, split_pairs):
    with open(path, "w") as f:
        for noisy, clean in split_pairs:
            f.write(json.dumps({"noisy": noisy, "clean": clean}) + "\n")

write_jsonl("data/llm_train.jsonl", train_pairs)
write_jsonl("data/llm_val.jsonl", val_pairs)

In [ ]:
import asyncio
import csv
import random

os.makedirs("data/audio", exist_ok=True)

import nest_asyncio
nest_asyncio.apply()

EDGE_VOICES = [
    "en-US-AriaNeural", "en-US-GuyNeural", "en-US-JennyNeural", "en-GB-SoniaNeural", "en-GB-RyanNeural", "en-AU-NatashaNeural", "en-IN-NeerjaNeural",
]

try:
    import edge_tts
    EDGE_TTS_AVAILABLE = True
except ImportError:
    EDGE_TTS_AVAILABLE = False
    print("edge-tts not installed/reachable -- falling back to gTTS (single voice).")

semaphore = asyncio.Semaphore(CONCURRENCY)

async def _synth_edge(text, voice, out_path):
    async with semaphore:
        communicate = edge_tts.Communicate(text, voice)
        await communicate.save(out_path)

def synthesize_one_gtts(text, out_path):
    from gtts import gTTS
    gTTS(text=text, lang="en").save(out_path)

async def synthesize_split_async(split_name, split_pairs):
    manifest_path = f"data/whisper_manifest_{split_name}.csv"
    rng = random.Random(42)

    tasks = []
    rows = []  # (audio_path, text, needs_gtts_fallback_flag placeholder)

    for i, (noisy, clean) in enumerate(split_pairs):
        audio_path = f"data/audio/{split_name}_{i:05d}.mp3"
        rows.append((audio_path, noisy))
        if not os.path.exists(audio_path) and EDGE_TTS_AVAILABLE:
            voice = rng.choice(EDGE_VOICES)
            tasks.append(_synth_edge(noisy, voice, audio_path))
        elif not os.path.exists(audio_path):
            synthesize_one_gtts(noisy, audio_path)

    if tasks:
        results = await asyncio.gather(*tasks, return_exceptions=True)
        for r in results:
            if isinstance(r, Exception):
                print("edge-tts task failed:", r)

    # gTTS fallback for any edge-tts tasks that failed and left no file
    for audio_path, noisy in rows:
        if not os.path.exists(audio_path):
            try:
                synthesize_one_gtts(noisy, audio_path)
            except Exception as error:
                print("gTTS fallback also failed for", audio_path, ":", error)

    with open(manifest_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["audio_path", "text"])
        for audio_path, noisy in rows:
            if os.path.exists(audio_path):
                writer.writerow([audio_path, noisy])

    print(f"[{split_name}] done: {sum(os.path.exists(p) for p, _ in rows)}/{len(rows)} synthesized")

asyncio.run(synthesize_split_async("train", train_pairs))
asyncio.run(synthesize_split_async("val", val_pairs))

## 5. Fine-tune Whisper (LoRA)

Fine-tune targets **robustness to messy/informal speech audio**, not content rewriting -- Whisper is trained to transcribe the noisy
sentence it hears **verbatim** (see the Section 4 note).

In [ ]:
!pip uninstall -y transformers tokenizers -q
!pip install -q "transformers==4.57.1" "tokenizers>=0.22,<0.24" "accelerate>=1.2" "safetensors>=0.5" "librosa>=0.10.2" "soundfile>=0.12.1"

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor

whisper_processor = WhisperProcessor.from_pretrained(WHISPER_BASE)

whisper_model = WhisperForConditionalGeneration.from_pretrained(
    WHISPER_BASE,
    attn_implementation="sdpa",
)

whisper_model.generation_config.forced_decoder_ids = None

In [ ]:
from peft import LoraConfig, get_peft_model

whisper_lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)

whisper_model = get_peft_model(
    whisper_model,
    whisper_lora_config
)

whisper_model.print_trainable_parameters()

In [ ]:
import librosa
import torch as torch_module  # avoid shadowing the torch import above

SAMPLING_RATE = 16000

class ManifestAudioDataset(torch_module.utils.data.Dataset):
    """Reads (audio_path, text) rows from a CSV and featurizes audio lazily."""

    def __init__(self, manifest_path, processor):
        self.rows = []
        with open(manifest_path) as f:
            reader = csv.DictReader(f)
            for row in reader:
                self.rows.append(row)
        self.processor = processor

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        audio, _ = librosa.load(row["audio_path"], sr=SAMPLING_RATE)

        input_features = self.processor.feature_extractor(
            audio, sampling_rate=SAMPLING_RATE
        ).input_features[0]

        labels = self.processor.tokenizer(row["text"]).input_ids

        return {"input_features": input_features, "labels": labels}

print("Dataset class ready.")

In [ ]:
class DataCollatorSpeechSeq2Seq:
    """Pads audio features and text labels separately for a training batch."""

    def __init__(self, processor):
        self.processor = processor

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all():
            labels = labels[:, 1:]

        batch["labels"] = labels

        # expected by the model's encoder.
        model_dtype = next(whisper_model.parameters()).dtype
        batch["input_features"] = batch["input_features"].to(dtype=model_dtype)
        
        return batch

whisper_collator = DataCollatorSpeechSeq2Seq(whisper_processor)
print("Collator ready.")

In [ ]:
whisper_train_ds = ManifestAudioDataset("data/whisper_manifest_train.csv", whisper_processor)
whisper_val_ds = ManifestAudioDataset("data/whisper_manifest_val.csv", whisper_processor)

print("Whisper train examples:", len(whisper_train_ds))
print("Whisper val examples:", len(whisper_val_ds))

In [ ]:
import evaluate

wer_metric = evaluate.load("wer")

def compute_wer(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = whisper_processor.tokenizer.pad_token_id

    pred_str = whisper_processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = whisper_processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

In [ ]:
from transformers import Seq2SeqTrainingArguments

num_update_steps_per_epoch = max(1, len(whisper_train_ds) // (WHISPER_BATCH_SIZE * 2))
total_train_steps = num_update_steps_per_epoch * WHISPER_EPOCHS
warmup_steps = int(0.1 * total_train_steps)

whisper_training_args = Seq2SeqTrainingArguments(
    output_dir=WHISPER_ADAPTER_DIR,
    per_device_train_batch_size=WHISPER_BATCH_SIZE,
    per_device_eval_batch_size=WHISPER_BATCH_SIZE,
    gradient_accumulation_steps=2,
    learning_rate=1e-3,
    warmup_steps=warmup_steps,
    num_train_epochs=WHISPER_EPOCHS,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    logging_steps=25,
    predict_with_generate=True,
    generation_max_length=225,
    report_to="none",
    label_names=["labels"],
    remove_unused_columns=False,
)

In [ ]:
from transformers import Seq2SeqTrainer

whisper_trainer = Seq2SeqTrainer(
    model=whisper_model,
    args=whisper_training_args,
    data_collator=whisper_collator,
    train_dataset=whisper_train_ds,
    eval_dataset=whisper_val_ds,
    processing_class=whisper_processor.feature_extractor,
    compute_metrics=compute_wer,
)

In [ ]:
whisper_trainer.train()

In [ ]:
whisper_model.save_pretrained(f"{WHISPER_ADAPTER_DIR}/final_adapter")
whisper_processor.save_pretrained(f"{WHISPER_ADAPTER_DIR}/final_adapter")

## 6. Fine-tune the LLM (QLoRA)

Same text pairs, now training the cleanup LLM. The base model is loaded in **4-bit** (QLoRA) so a 7B model fits on a single T4 during training.

In [ ]:
from transformers import AutoTokenizer

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_BASE)

if llm_tokenizer.pad_token is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token

In [ ]:
CLEANUP_SYSTEM_PROMPT = (
    "You are a transcription cleanup assistant. Rewrite the raw speech "
    "transcript into clear, well-punctuated text. Remove filler words and "
    "false starts, fix grammar. Do not add information that was not said. "
    "Output ONLY the cleaned text."
)

def build_chat_example(row):
    messages = [
        {"role": "system", "content": CLEANUP_SYSTEM_PROMPT},
        {"role": "user", "content": row["noisy"]},
        {"role": "assistant", "content": row["clean"]},
    ]
    text = llm_tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

llm_train_rows = [build_chat_example(r) for r in load_jsonl("data/llm_train.jsonl")]
llm_val_rows = [build_chat_example(r) for r in load_jsonl("data/llm_val.jsonl")]

print("LLM train examples:", len(llm_train_rows))
print("LLM val examples:", len(llm_val_rows))

In [ ]:
from datasets import Dataset

MAX_LENGTH = 512

def tokenize_fn(examples):
    return llm_tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

llm_train_ds = Dataset.from_list(llm_train_rows).map(tokenize_fn, batched=True, remove_columns=["text"])
llm_val_ds = Dataset.from_list(llm_val_rows).map(tokenize_fn, batched=True, remove_columns=["text"])

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_BASE,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
)

llm_model = prepare_model_for_kbit_training(llm_model)

In [ ]:
llm_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

llm_model = get_peft_model(llm_model, llm_lora_config)
llm_model.print_trainable_parameters()

In [ ]:
from transformers import DataCollatorForLanguageModeling, TrainingArguments, Trainer

llm_collator = DataCollatorForLanguageModeling(
    tokenizer=llm_tokenizer,
    mlm=False,
)

llm_training_args = TrainingArguments(
    output_dir=LLM_ADAPTER_DIR,
    num_train_epochs=LLM_EPOCHS,
    per_device_train_batch_size=LLM_BATCH_SIZE,
    per_device_eval_batch_size=LLM_BATCH_SIZE,
    gradient_accumulation_steps=4,
    optim="adamw_torch",
    learning_rate=2e-4,
    weight_decay=0.01,
    adam_beta1=0.9,
    adam_beta2=0.999,
    adam_epsilon=1e-8,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    max_grad_norm=1.0,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    logging_strategy="steps",
    logging_steps=10,
    logging_first_step=True,
    report_to="none",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    seed=42,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    remove_unused_columns=True,
)

In [ ]:
llm_trainer = Trainer(
    model=llm_model,
    args=llm_training_args,
    train_dataset=llm_train_ds,
    eval_dataset=llm_val_ds,
    data_collator=llm_collator,
)

In [ ]:
llm_trainer.train()

In [ ]:
llm_model.save_pretrained(f"{LLM_ADAPTER_DIR}/final_adapter")
llm_tokenizer.save_pretrained(f"{LLM_ADAPTER_DIR}/final_adapter")

## 7. Custom CUDA kernel (training-speed optimization)

A hand-written kernel that fuses "add bias" + "GELU activation" into one pass, instead of two separate elementwise ops. Elementwise ops like this are memory-bandwidth bound, not compute bound — fusing them halves how many times each value is read/written from GPU memory, which is where the speedup actually comes from.

In [ ]:
cuda_kernel_source = r'''
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>
#include <math.h>

#define SQRT_2_OVER_PI 0.7978845608028654f
#define GELU_COEF 0.044715f

__device__ __forceinline__ float gelu_fwd(float x) {
    float x3 = x * x * x;
    float inner = SQRT_2_OVER_PI * (x + GELU_COEF * x3);
    return 0.5f * x * (1.0f + tanhf(inner));
}

__device__ __forceinline__ float gelu_bwd(float x) {
    float x2 = x * x;
    float x3 = x2 * x;
    float inner = SQRT_2_OVER_PI * (x + GELU_COEF * x3);
    float tanh_inner = tanhf(inner);
    float sech2 = 1.0f - tanh_inner * tanh_inner;
    float d_inner = SQRT_2_OVER_PI * (1.0f + 3.0f * GELU_COEF * x2);
    return 0.5f * (1.0f + tanh_inner) + 0.5f * x * sech2 * d_inner;
}

__global__ void fused_bias_gelu_fwd_kernel(
    const float* __restrict__ x, const float* __restrict__ bias,
    float* __restrict__ y, int rows, int cols
) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    int total = rows * cols;
    for (int i = idx; i < total; i += stride) {
        int col = i % cols;
        float val = x[i] + bias[col];
        y[i] = gelu_fwd(val);
    }
}

__global__ void fused_bias_gelu_bwd_kernel(
    const float* __restrict__ grad_out, const float* __restrict__ x,
    const float* __restrict__ bias, float* __restrict__ grad_x,
    float* __restrict__ grad_bias, int rows, int cols
) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    int total = rows * cols;
    for (int i = idx; i < total; i += stride) {
        int col = i % cols;
        float val = x[i] + bias[col];
        float local_grad = gelu_bwd(val) * grad_out[i];
        grad_x[i] = local_grad;
        atomicAdd(&grad_bias[col], local_grad);
    }
}

torch::Tensor fused_bias_gelu_forward(torch::Tensor x, torch::Tensor bias) {
    x = x.contiguous();
    bias = bias.contiguous();
    int rows = x.size(0);
    int cols = x.size(1);
    auto y = torch::empty_like(x);
    int threads = 256;
    int total = rows * cols;
    int blocks = std::min((total + threads - 1) / threads, 65535);
    fused_bias_gelu_fwd_kernel<<<blocks, threads>>>(
        x.data_ptr<float>(), bias.data_ptr<float>(), y.data_ptr<float>(), rows, cols
    );
    return y;
}

std::vector<torch::Tensor> fused_bias_gelu_backward(
    torch::Tensor grad_out, torch::Tensor x, torch::Tensor bias
) {
    grad_out = grad_out.contiguous();
    x = x.contiguous();
    bias = bias.contiguous();
    int rows = x.size(0);
    int cols = x.size(1);
    auto grad_x = torch::empty_like(x);
    auto grad_bias = torch::zeros_like(bias);
    int threads = 256;
    int total = rows * cols;
    int blocks = std::min((total + threads - 1) / threads, 65535);
    fused_bias_gelu_bwd_kernel<<<blocks, threads>>>(
        grad_out.data_ptr<float>(), x.data_ptr<float>(), bias.data_ptr<float>(),
        grad_x.data_ptr<float>(), grad_bias.data_ptr<float>(), rows, cols
    );
    return {grad_x, grad_bias};
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("forward", &fused_bias_gelu_forward, "Fused bias+GELU forward (CUDA)");
    m.def("backward", &fused_bias_gelu_backward, "Fused bias+GELU backward (CUDA)");
}
'''

with open("fused_bias_gelu_kernel.cu", "w") as f:
    f.write(cuda_kernel_source)

In [ ]:
from torch.utils.cpp_extension import load

fused_bias_gelu_ext = load(
    name="fused_bias_gelu_ext",
    sources=["fused_bias_gelu_kernel.cu"],
    verbose=True,
)

In [ ]:
class FusedBiasGELU(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, bias):
        y = fused_bias_gelu_ext.forward(x, bias)
        ctx.save_for_backward(x, bias)
        return y

    @staticmethod
    def backward(ctx, grad_output):
        x, bias = ctx.saved_tensors
        grad_x, grad_bias = fused_bias_gelu_ext.backward(grad_output, x, bias)
        return grad_x, grad_bias

def fused_bias_gelu(x, bias):
    return FusedBiasGELU.apply(x, bias)

In [ ]:
import torch.nn.functional as F

def torch_reference(x, bias):
    return F.gelu(x + bias, approximate="tanh")

torch.manual_seed(0)
x = torch.randn(64, 1024, device="cuda", requires_grad=True)
bias = torch.randn(1024, device="cuda", requires_grad=True)

x_ref = x.detach().clone().requires_grad_(True)
bias_ref = bias.detach().clone().requires_grad_(True)

y_custom = fused_bias_gelu(x, bias)
y_ref = torch_reference(x_ref, bias_ref)

forward_matches = torch.allclose(y_custom, y_ref, atol=1e-4, rtol=1e-4)
print("Forward output matches PyTorch:", forward_matches)

grad_out = torch.randn_like(y_custom)
y_custom.backward(grad_out)
y_ref.backward(grad_out)

grad_x_matches = torch.allclose(x.grad, x_ref.grad, atol=1e-4, rtol=1e-4)
grad_bias_matches = torch.allclose(bias.grad, bias_ref.grad, atol=1e-3, rtol=1e-3)
print("grad_x matches PyTorch:", grad_x_matches)
print("grad_bias matches PyTorch:", grad_bias_matches)

In [ ]:
import time

def benchmark_op(fn, x, bias, iters=50):
    torch.cuda.synchronize()
    start = time.time()
    for _ in range(iters):
        x.grad = None
        bias.grad = None
        y = fn(x, bias)
        y.backward(torch.ones_like(y))
    torch.cuda.synchronize()
    return (time.time() - start) / iters * 1000  # milliseconds per iteration

x = torch.randn(4096, 4096, device="cuda", requires_grad=True)
bias = torch.randn(4096, device="cuda", requires_grad=True)

benchmark_op(fused_bias_gelu, x, bias, iters=5)
benchmark_op(torch_reference, x, bias, iters=5)

custom_ms = benchmark_op(fused_bias_gelu, x, bias)
reference_ms = benchmark_op(torch_reference, x, bias)

print(f"Custom fused kernel: {custom_ms:.3f} ms/iter")
print(f"PyTorch (add + gelu): {reference_ms:.3f} ms/iter")
print(f"Speedup: {reference_ms / custom_ms:.2f}x")

## 8. Merge adapters for low-latency serving

LoRA adapters add a small amount of compute overhead at inference (an extra matmul per adapted layer). **Merging** folds the adapter weights directly into the base model, so serving runs at the same speed as the original base model — this matters for a low-latency target.

In [ ]:
from peft import PeftModel

whisper_base_for_merge = WhisperForConditionalGeneration.from_pretrained(WHISPER_BASE)
whisper_merged = PeftModel.from_pretrained(whisper_base_for_merge, f"{WHISPER_ADAPTER_DIR}/final_adapter")
whisper_merged = whisper_merged.merge_and_unload()

os.makedirs(WHISPER_MERGED_DIR, exist_ok=True)
whisper_merged.save_pretrained(WHISPER_MERGED_DIR)
whisper_processor.save_pretrained(WHISPER_MERGED_DIR)

In [ ]:
llm_base_fp16 = AutoModelForCausalLM.from_pretrained(
    LLM_BASE,
    torch_dtype=torch.float16,
    device_map="auto",
    attn_implementation="sdpa",
)

llm_merged = PeftModel.from_pretrained(llm_base_fp16, f"{LLM_ADAPTER_DIR}/final_adapter")
llm_merged = llm_merged.merge_and_unload()

os.makedirs(LLM_MERGED_DIR, exist_ok=True)
llm_merged.save_pretrained(LLM_MERGED_DIR)
llm_tokenizer.save_pretrained(LLM_MERGED_DIR)

print("Saved merged LLM to", LLM_MERGED_DIR)

## 9. End-to-end latency check

A quick sanity check that the merged, serving-ready pipeline is actually fast: transcribe a short audio clip, then generate the cleaned text, and time both stages.


In [ ]:
from transformers import pipeline, TextIteratorStreamer

asr_pipe = pipeline(
    "automatic-speech-recognition",
    model=whisper_merged,
    tokenizer=whisper_processor.tokenizer,
    feature_extractor=whisper_processor.feature_extractor,
    torch_dtype=torch.float16,
    device="cuda:0",
)

test_audio_path = val_pairs and f"data/audio/val_00000.mp3"

start = time.time()
asr_result = asr_pipe(test_audio_path)
asr_seconds = time.time() - start

print("Raw transcript:", asr_result["text"])
print(f"ASR time: {asr_seconds * 1000:.0f} ms")

In [ ]:
messages = [
    {"role": "system", "content": CLEANUP_SYSTEM_PROMPT},
    {"role": "user", "content": asr_result["text"]},
]

prompt = llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = llm_tokenizer(prompt, return_tensors="pt").to(llm_merged.device)

start = time.time()
output_ids = llm_merged.generate(
    **inputs,
    max_new_tokens=256,
    do_sample=False,
    repetition_penalty=1.1,
)
llm_seconds = time.time() - start

new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
cleaned_text = llm_tokenizer.decode(new_tokens, skip_special_tokens=True)

print("Cleaned text:", cleaned_text)
print(f"LLM time: {llm_seconds * 1000:.0f} ms  ({len(new_tokens) / llm_seconds:.1f} tokens/sec)")
print(f"Total end-to-end: {(asr_seconds + llm_seconds) * 1000:.0f} ms")

## 10. Push the merged models to Hugging Face Hub

In [ ]:
from huggingface_hub import login
login(token="hf_xxx")

In [ ]:
HF_USERNAME = "aijadugar"
WHISPER_REPO_ID = f"{HF_USERNAME}/wisprflow-clone-whisper"
LLM_REPO_ID = f"{HF_USERNAME}/wisprflow-clone-llm"

whisper_merged.push_to_hub(WHISPER_REPO_ID)
whisper_processor.push_to_hub(WHISPER_REPO_ID)

In [ ]:
llm_merged.push_to_hub(LLM_REPO_ID)
llm_tokenizer.push_to_hub(LLM_REPO_ID)

## 11. Free local disk

In [ ]:
import shutil

for path in [WHISPER_MERGED_DIR, LLM_MERGED_DIR, WHISPER_ADAPTER_DIR, LLM_ADAPTER_DIR, "data/audio"]:
    if os.path.exists(path):
        shutil.rmtree(path)
        print("Removed", path)

!df -h /kaggle/working 2>/dev/null || df -h .